In [0]:
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.setup")
print("Volume ready: /Volumes/workspace/default/setup")

In [0]:
import glob, os, json
from databricks.sdk import WorkspaceClient

SCOPE = "power-prices"
KEY = "gcp-sa-json"
VOLUME_DIR = "/Volumes/workspace/default/setup"

w = WorkspaceClient()

# 1. Find the uploaded key file
files = glob.glob(f"{VOLUME_DIR}/*.json")
assert len(files) == 1, f"Expected exactly one JSON file, found: {files}"
with open(files[0]) as f:
    sa_json = f.read()
print("Key file found for:", json.loads(sa_json)["client_email"])

# 2. Create the scope (once) and store the key
if SCOPE not in [s.name for s in w.secrets.list_scopes()]:
    w.secrets.create_scope(scope=SCOPE)
    print(f"Created secret scope: {SCOPE}")
w.secrets.put_secret(scope=SCOPE, key=KEY, string_value=sa_json)
print(f"Stored secret: {SCOPE}/{KEY}")

# 3. Delete the file from the volume
os.remove(files[0])
print("Files left in volume:", glob.glob(f"{VOLUME_DIR}/*"))

# 4. Verify by reading it back (shows the email only, never the key)
check = json.loads(dbutils.secrets.get(SCOPE, KEY))
print("Secret readable. Service account:", check["client_email"])